# 第6章 追加学習の判断と実践

この章では、continued pretraining の考え方を小さく体験します。第5章の supervised LoRA は「指示に対する出力形式を安定させる」学習でした。第6章では、短い教材コーパスを使い、次トークン予測でモデルを追加学習します。

RTX 4060 Ti 16GB で安全に試すため、ここでもベースモデル全体は更新せず、LoRA adapter による continued-pretraining 風の追加学習にします。実務で本格的に続ける前に、評価セット、保存先、rollback を決める必要があります。


In [ ]:
from pathlib import Path
import os
import json
import sys

# Notebook をどこから開いても helper を import できるようにします。
search_roots = [Path.cwd()]
env_root = os.environ.get("LOCAL_LLM_REPO_ROOT")
if env_root:
    search_roots.append(Path(env_root))
search_roots.append(Path("C:/LLM"))

seen = set()
for root in search_roots:
    current = root.resolve()
    for candidate in [current, *current.parents]:
        if candidate in seen:
            continue
        seen.add(candidate)
        helper_dir = candidate / "notebooks"
        if (helper_dir / "local_llm_practice.py").exists():
            sys.path.insert(0, str(helper_dir))
            break
    else:
        continue
    break
else:
    raise RuntimeError(
        "notebooks/local_llm_practice.py が見つかりません。"
        "C:/LLM か notebooks/ 配下で開くか、LOCAL_LLM_REPO_ROOT を設定してください。"
    )

from local_llm_practice import (
    DATA_DIR,
    DOCS_DIR,
    REPO_ROOT,
    WORK_DIR,
    TrainingConfig,
    ask_about_chapter,
    attach_lora,
    configure_local_caches,
    count_trainable_parameters,
    generate_text,
    gpu_summary,
    load_base_model,
    load_chapter,
    load_jsonl,
    make_cpt_features,
    make_sft_features,
    nvidia_smi_summary,
    ollama_generate,
    print_headings,
    read_text,
    retrieve_chunks,
    split_markdown,
    train_lora_adapter,
)

configure_local_caches()
print("REPO_ROOT:", REPO_ROOT)
print("DOCS_DIR :", DOCS_DIR)
print("WORK_DIR :", WORK_DIR)

chapter_path, chapter_text = load_chapter("06-continued-pretraining.md")
print(chapter_path)
print_headings(chapter_text)


## 1. 第5章との違いを確認する

第5章は instruction/input/output のペアを使う supervised fine-tuning です。第6章は、コーパスの続きを予測する形で、語彙や文体への適応を小さく試します。


In [ ]:
print(ask_about_chapter(
    chapter_text,
    "第5章の LoRA / QLoRA と、第6章の継続事前学習の違いを、教材実践の観点で説明してください。",
))


## 2. GPU と学習設定を確認する

第6章も RTX 4060 Ti 16GB で実行できるよう、短い系列長と少ない step にします。


In [ ]:
print(nvidia_smi_summary())
print(json.dumps(gpu_summary(), ensure_ascii=False, indent=2))

config = TrainingConfig(
    model_id="Qwen/Qwen2.5-0.5B-Instruct",
    max_length=160,
    max_steps=20,
    learning_rate=1e-4,
    lora_r=8,
    lora_alpha=16,
)
print(config)


## 3. 追加学習用の教材コーパスを読む

ここでは公開可否を確認済みの短い教材テキストだけを使います。自分の作業データで試す場合は、この repository ではなく git 管理外の場所へ置きます。


In [ ]:
corpus_text = read_text(DATA_DIR / "continued_pretraining_corpus.txt")
print(corpus_text)


## 4. 事前学習済みモデルを読み、学習前の応答を見る


In [ ]:
tokenizer, base_model = load_base_model(config)
probe_prompt = "ローカル LLM 活用で追加学習を考える前に確認することを3点で説明してください。"
before_text = generate_text(base_model, tokenizer, probe_prompt, max_new_tokens=96)
print(before_text)


## 5. continued-pretraining 風の LoRA adapter を学習する

`make_cpt_features` はコーパスを短い token 列に分け、次トークン予測の labels を作ります。これは instruction/output の正解を覚える第5章とは違い、文体や語彙への適応を小さく試すための学習です。


In [ ]:
cpt_model = attach_lora(base_model, config)
features = make_cpt_features([corpus_text], tokenizer, max_length=config.max_length)
adapter_dir = WORK_DIR / "chapter06-continued-pretraining-adapter"

metrics = train_lora_adapter(
    cpt_model,
    tokenizer,
    features,
    adapter_dir,
    config,
)
print(json.dumps(metrics, ensure_ascii=False, indent=2))


## 6. 学習後の応答と前後比較

短い教材コーパスだけでは大きな品質改善を狙いません。ここで見るのは、実際に追加学習が走り、adapter が保存され、同じ評価プロンプトで比較できる状態になったかです。


In [ ]:
after_text = generate_text(cpt_model, tokenizer, probe_prompt, max_new_tokens=96)
print("=== before ===")
print(before_text)
print("\n=== after ===")
print(after_text)


## 7. 本格実行へ進む前の判断

短い実験で手順を確認できたら、評価セット、入力範囲、保存先、rollback を決めます。この確認なしに本格的な追加学習へ進むと、良くなったのか悪くなったのか判断できません。


In [ ]:
planning_prompt = """
次の短い continued-pretraining 風実験を、本格運用に進める前に確認すべきことを checklist にしてください。
観点: 評価セット、入力データの公開可否、保存先、rollback、RAGやプロンプトで代替できない理由。
""".strip()
print(ollama_generate(planning_prompt, temperature=0.2))
print("\nadapter files:")
for path in sorted(adapter_dir.glob("*")):
    print(path)
